In [17]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, top_k_accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import lightgbm as lgb
import pickle

In [2]:
df=pd.read_csv(r"D:\ml_udemy\00 ML CODE\Projects\Digital Diagnosis v2\data\processed\processed_dataset.csv")

In [4]:
df.head()

,diseases,anxiety_and_nervousness,depression,shortness_of_breath,depressive_or_psychotic_symptoms,sharp_chest_pain,dizziness,insomnia,abnormal_involuntary_movements,chest_tightness,...,stuttering_or_stammering,problems_with_orgasm,nose_deformity,lump_over_jaw,sore_in_nose,hip_weakness,back_swelling,ankle_stiffness_or_tightness,ankle_weakness,neck_weakness
0,panic disorder,1,0,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,panic disorder,0,0,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,panic disorder,1,1,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,panic disorder,1,0,0,1,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,panic disorder,1,1,0,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0


In [5]:
x=df.drop("diseases", axis=1)
y=df['diseases']

In [6]:
le=LabelEncoder()
y=le.fit_transform(y)

In [7]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, stratify=y, random_state=42)

In [8]:
model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=127,
    min_child_samples=10,
    feature_fraction=0.5,
    bagging_fraction=0.8,
    bagging_freq=5,
    max_depth=20,
    reg_alpha=0.5,
    reg_lambda=0.5,
    class_weight='balanced',
    random_state=42,
    verbose=-1,
    n_jobs=-1
)

In [9]:
model.fit(x_train, y_train)

,boosting_type,'gbdt'
,num_leaves,127
,max_depth,20
,learning_rate,0.05
,n_estimators,1000
,subsample_for_bin,200000
,objective,None
,class_weight,'balanced'
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,10


In [10]:
y_pred=model.predict(x_test)
y_prob = model.predict_proba(x_test)

In [11]:
print("Top-1:", accuracy_score(y_test, y_pred))
print("Top-3:", top_k_accuracy_score(y_test, y_prob, k=3))
print("Top-5:", top_k_accuracy_score(y_test, y_prob, k=5))

print("Macro Precision:", precision_score(y_test, y_pred, average='macro'))
print("Macro Recall:", recall_score(y_test, y_pred, average='macro'))
print("Macro F1:", f1_score(y_test, y_pred, average='macro'))
print("Weighted F1:", f1_score(y_test, y_pred, average='weighted'))

print("\nFull Classification Report:\n")
print(classification_report(y_test, y_pred))

Top-1: 0.8575313154727411
Top-3: 0.9597913787723443
Top-5: 0.981277582133464
Macro Precision: 0.8535752357809453
Macro Recall: 0.8773144986710962
Macro F1: 0.8610356784778864
Weighted F1: 0.8582075446622126

Full Classification Report:

              precision    recall  f1-score   support

           0       0.95      0.99      0.97        81
           1       0.70      0.84      0.77        58
           2       0.88      0.97      0.92        68
           3       0.62      0.79      0.70        99
           4       0.88      0.70      0.78       182
           5       0.97      0.89      0.93       241
           6       0.85      0.66      0.74       243
           7       0.63      0.75      0.69       181
           8       0.28      0.40      0.33        42
           9       0.98      0.95      0.97       182
          10       0.69      0.84      0.76       137
          11       0.95      0.81      0.88       241
          12       0.86      0.94      0.90        90
      

In [12]:
with open(r"D:\ml_udemy\00 ML CODE\Projects\Digital Diagnosis v2\models\lightgbm_model.pkl", "wb") as f:
    pickle.dump(model, f)

with open(r"D:\ml_udemy\00 ML CODE\Projects\Digital Diagnosis v2\models\label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

print("Model saved!")

Model saved!


In [15]:
def predict_top3(symptom_values: list):
    """Pass a list of 0/1 values matching your symptom columns."""
    x = np.array(symptom_values).reshape(1, -1)
    probs = model.predict_proba(x)[0]
    top3_idx = np.argsort(probs)[::-1][:3]

    print("Top 3 Predicted Diseases:")
    for rank, idx in enumerate(top3_idx, 1):
        disease = le.inverse_transform([idx])[0]
        prob = probs[idx] * 100
        print(f"  {rank}. {disease:20s} {prob:.2f}%")

In [16]:
sample_input = [0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]
predict_top3(sample_input)

Top 3 Predicted Diseases:
  1. acute respiratory distress syndrome (ards) 89.37%
  2. pneumonia            10.50%
  3. asthma               0.05%


e:\alternativeofC\enviorenments\MLenv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
